# Exploratory Data Analysis — Property Climate Risk

This notebook explores the synthetic property + climate dataset used to train the
XGBoost valuation model. It covers the schema, target distribution, flood-zone mix,
elevation vs. hazard relationships, feature correlations, and the composite climate-
risk score.

Run from the repo root so that `src` is importable.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.pipeline import ingest, features as feat

pd.set_option('display.max_columns', 40)
df = ingest.load_data()
print(df.shape)
df.head()

In [ ]:
df.describe(include='all').T

## Target distribution & parish breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df['property_value_usd'].plot.hist(bins=60, ax=axes[0], title='Property value (USD)')
df.groupby('county')['property_value_usd'].median().plot.bar(ax=axes[1], title='Median value by parish')
plt.tight_layout()
plt.show()

## Flood-zone mix and elevation relationship

In [ ]:
print(df['fema_flood_zone'].value_counts(normalize=True).round(3))
df.boxplot(column='elevation_ft', by='fema_flood_zone', figsize=(8, 4))
plt.title('Elevation by FEMA flood zone')
plt.suptitle('')
plt.show()

## Feature correlations with property value

In [ ]:
numeric = df.select_dtypes('number')
corr = numeric.corr()['property_value_usd'].sort_values(ascending=False)
corr

## Composite climate-risk score

In [ ]:
engineered = feat.build_features(df)
engineered['climate_risk_score'] = feat.compute_climate_risk_score(engineered)
engineered['risk_band'] = engineered['climate_risk_score'].apply(feat.risk_band)
print(engineered['risk_band'].value_counts())
engineered['climate_risk_score'].plot.hist(bins=50, figsize=(8, 4), title='Composite climate-risk score')
plt.show()

## Spatial scatter (parcels coloured by risk)

In [ ]:
sample = engineered.sample(4000, random_state=0)
plt.figure(figsize=(8, 7))
sc = plt.scatter(sample['longitude'], sample['latitude'], c=sample['climate_risk_score'],
                 cmap='RdYlGn_r', s=6, alpha=0.6)
plt.colorbar(sc, label='Climate-risk score')
plt.xlabel('Longitude'); plt.ylabel('Latitude'); plt.title('Parcel climate risk (sample)')
plt.show()